In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

In [2]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [3]:
device = 'cuda'

In [4]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
# X_test = scaler.transform(X_test)

X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)

y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)

In [5]:
train_data = TensorDataset(X_train, y_train)
valid_data = TensorDataset(X_valid, y_valid)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32, shuffle=False)

In [6]:
class CancerClassifier(nn.Module):
    def __init__(self, n_features, n_units_l1, n_units_l2, dropout_rate):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features, n_units_l1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(n_units_l1, n_units_l2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
        )
        self.output_layer = nn.Linear(n_units_l2, 1)
    
    def forward(self, X):
        deep = self.deep_stack(X)
        return self.output_layer(deep)

In [7]:
def train(model, optimizer, criterion, train_loader, valid_loader, n_epochs):
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_train_loss = total_loss / len(train_loader)

        model.eval()
        total_valid_loss = 0
        total_correct = 0      # New: Keep track of right answers
        total_samples = 0      # New: Keep track of total patients tested
        
        with torch.no_grad():
            for X_batch, y_batch in valid_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                
                # 1. Get raw predictions and calculate loss
                raw_pred = model(X_batch)
                loss = criterion(raw_pred, y_batch)
                total_valid_loss += loss.item()
                
                # 2. ACCURACY MATH: Convert raw numbers to probabilities (0 to 1)
                probabilities = torch.sigmoid(raw_pred)
                
                # 3. Round to 0 or 1 (If > 0.5, guess 1. Else guess 0)
                guesses = probabilities.round()
                
                # 4. Count how many guesses match the actual y_batch
                correct_guesses = (guesses == y_batch).sum().item()
                
                total_correct += correct_guesses
                total_samples += len(y_batch) # Add the number of patients in this batch
                
        mean_valid_loss = total_valid_loss / len(valid_loader)
        
        # Calculate final accuracy percentage for the epoch
        accuracy = (total_correct / total_samples) * 100
        
        # Print the report!
        print(f"Epoch {epoch + 1:02d}/{n_epochs} | Train Loss: {mean_train_loss:.4f} | Valid Loss: {mean_valid_loss:.4f} | Accuracy: {accuracy:.2f}%")
    return accuracy

In [8]:
import optuna
def objective(trial):
    l1_neurons = trial.suggest_int("l1_neurons", 16, 64)
    l2_neurons = trial.suggest_int("l2_neurons", 4, 32)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)

    model = CancerClassifier(
        n_features=30,
        n_units_l1=l1_neurons,
        n_units_l2=l2_neurons,
        dropout_rate=dropout_rate
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    final_score = train(
        model=model,
        optimizer=optimizer,
        criterion=criterion,
        train_loader=train_loader,
        valid_loader=valid_loader,
        n_epochs=20
    )
    return final_score


In [9]:
study = optuna.create_study(direction="maximize")
print("Starting Optuna search...")

study.optimize(objective, n_trials=20)

print("\n---WINNING SETUP---")
print(f"Best Accuracy: {study.best_value:.2f}%")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-04-20 18:37:55,795] A new study created in memory with name: no-name-09091786-8852-4773-b321-55a2b8795c8b


Starting Optuna search...
Epoch 01/20 | Train Loss: 0.6255 | Valid Loss: 0.4989 | Accuracy: 98.25%
Epoch 02/20 | Train Loss: 0.4132 | Valid Loss: 0.2604 | Accuracy: 95.61%
Epoch 03/20 | Train Loss: 0.2550 | Valid Loss: 0.1375 | Accuracy: 97.37%
Epoch 04/20 | Train Loss: 0.1861 | Valid Loss: 0.0889 | Accuracy: 99.12%
Epoch 05/20 | Train Loss: 0.1398 | Valid Loss: 0.0718 | Accuracy: 99.12%
Epoch 06/20 | Train Loss: 0.1389 | Valid Loss: 0.0651 | Accuracy: 98.25%
Epoch 07/20 | Train Loss: 0.1176 | Valid Loss: 0.0620 | Accuracy: 98.25%
Epoch 08/20 | Train Loss: 0.1138 | Valid Loss: 0.0582 | Accuracy: 99.12%
Epoch 09/20 | Train Loss: 0.1032 | Valid Loss: 0.0567 | Accuracy: 99.12%
Epoch 10/20 | Train Loss: 0.0845 | Valid Loss: 0.0537 | Accuracy: 99.12%
Epoch 11/20 | Train Loss: 0.1014 | Valid Loss: 0.0517 | Accuracy: 99.12%
Epoch 12/20 | Train Loss: 0.0995 | Valid Loss: 0.0513 | Accuracy: 99.12%
Epoch 13/20 | Train Loss: 0.1037 | Valid Loss: 0.0545 | Accuracy: 98.25%
Epoch 14/20 | Train Loss:

[I 2026-04-20 18:38:00,743] Trial 0 finished with value: 98.24561403508771 and parameters: {'l1_neurons': 36, 'l2_neurons': 6, 'dropout_rate': 0.3181015216626728, 'lr': 0.002903329552221925}. Best is trial 0 with value: 98.24561403508771.


Epoch 20/20 | Train Loss: 0.0703 | Valid Loss: 0.0545 | Accuracy: 98.25%
Epoch 01/20 | Train Loss: 0.5459 | Valid Loss: 0.3245 | Accuracy: 95.61%
Epoch 02/20 | Train Loss: 0.2239 | Valid Loss: 0.0804 | Accuracy: 97.37%
Epoch 03/20 | Train Loss: 0.1216 | Valid Loss: 0.0526 | Accuracy: 97.37%
Epoch 04/20 | Train Loss: 0.0910 | Valid Loss: 0.0567 | Accuracy: 97.37%
Epoch 05/20 | Train Loss: 0.0743 | Valid Loss: 0.0552 | Accuracy: 98.25%
Epoch 06/20 | Train Loss: 0.0718 | Valid Loss: 0.0494 | Accuracy: 99.12%
Epoch 07/20 | Train Loss: 0.0537 | Valid Loss: 0.0506 | Accuracy: 99.12%
Epoch 08/20 | Train Loss: 0.0593 | Valid Loss: 0.0538 | Accuracy: 99.12%
Epoch 09/20 | Train Loss: 0.0696 | Valid Loss: 0.0517 | Accuracy: 99.12%
Epoch 10/20 | Train Loss: 0.0424 | Valid Loss: 0.0572 | Accuracy: 97.37%
Epoch 11/20 | Train Loss: 0.0632 | Valid Loss: 0.0567 | Accuracy: 98.25%
Epoch 12/20 | Train Loss: 0.0519 | Valid Loss: 0.0621 | Accuracy: 98.25%
Epoch 13/20 | Train Loss: 0.0507 | Valid Loss: 0.05

[I 2026-04-20 18:38:01,596] Trial 1 finished with value: 97.36842105263158 and parameters: {'l1_neurons': 23, 'l2_neurons': 17, 'dropout_rate': 0.291588316907594, 'lr': 0.005092045487918384}. Best is trial 0 with value: 98.24561403508771.


...

Epoch 14/20 | Train Loss: 0.3699 | Valid Loss: 0.3075 | Accuracy: 96.49%
Epoch 15/20 | Train Loss: 0.3268 | Valid Loss: 0.2729 | Accuracy: 96.49%
Epoch 16/20 | Train Loss: 0.2997 | Valid Loss: 0.2410 | Accuracy: 96.49%
Epoch 17/20 | Train Loss: 0.2766 | Valid Loss: 0.2133 | Accuracy: 96.49%
Epoch 18/20 | Train Loss: 0.2426 | Valid Loss: 0.1892 | Accuracy: 96.49%
Epoch 19/20 | Train Loss: 0.2364 | Valid Loss: 0.1697 | Accuracy: 97.37%
Epoch 20/20 | Train Loss: 0.2032 | Valid Loss: 0.1532 | Accuracy: 97.37%
Epoch 01/20 | Train Loss: 0.6369 | Valid Loss: 0.5493 | Accuracy: 95.61%
Epoch 02/20 | Train Loss: 0.4958 | Valid Loss: 0.3515 | Accuracy: 95.61%
Epoch 03/20 | Train Loss: 0.3142 | Valid Loss: 0.1702 | Accuracy: 96.49%
Epoch 04/20 | Train Loss: 0.2072 | Valid Loss: 0.0919 | Accuracy: 97.37%
Epoch 05/20 | Train Loss: 0.1570 | Valid Loss: 0.0671 | Accuracy: 98.25%
Epoch 06/20 | Train Loss: 0.1507 | Valid Loss: 0.0587 | Accuracy: 97.37%
Epoch 07/20 | Train Loss: 0.0929 | Valid Loss: 0.05

[I 2026-04-20 18:38:12,832] Trial 18 finished with value: 98.24561403508771 and parameters: {'l1_neurons': 27, 'l2_neurons': 20, 'dropout_rate': 0.4042271020784788, 'lr': 0.002522982675875845}. Best is trial 5 with value: 99.12280701754386.


Epoch 15/20 | Train Loss: 0.0585 | Valid Loss: 0.0481 | Accuracy: 98.25%
Epoch 16/20 | Train Loss: 0.0544 | Valid Loss: 0.0464 | Accuracy: 99.12%
Epoch 17/20 | Train Loss: 0.0565 | Valid Loss: 0.0434 | Accuracy: 99.12%
Epoch 18/20 | Train Loss: 0.0494 | Valid Loss: 0.0437 | Accuracy: 99.12%
Epoch 19/20 | Train Loss: 0.0396 | Valid Loss: 0.0463 | Accuracy: 98.25%
Epoch 20/20 | Train Loss: 0.0498 | Valid Loss: 0.0476 | Accuracy: 98.25%
Epoch 01/20 | Train Loss: 0.2377 | Valid Loss: 0.1162 | Accuracy: 95.61%
Epoch 02/20 | Train Loss: 0.1129 | Valid Loss: 0.0549 | Accuracy: 98.25%
Epoch 03/20 | Train Loss: 0.0590 | Valid Loss: 0.0616 | Accuracy: 98.25%
Epoch 04/20 | Train Loss: 0.0467 | Valid Loss: 0.0737 | Accuracy: 97.37%
Epoch 05/20 | Train Loss: 0.0435 | Valid Loss: 0.0772 | Accuracy: 98.25%
Epoch 06/20 | Train Loss: 0.0475 | Valid Loss: 0.2200 | Accuracy: 94.74%
Epoch 07/20 | Train Loss: 0.0384 | Valid Loss: 0.1017 | Accuracy: 98.25%
Epoch 08/20 | Train Loss: 0.0401 | Valid Loss: 0.20

[I 2026-04-20 18:38:13,530] Trial 19 finished with value: 97.36842105263158 and parameters: {'l1_neurons': 36, 'l2_neurons': 15, 'dropout_rate': 0.17050354238989746, 'lr': 0.03354456173108639}. Best is trial 5 with value: 99.12280701754386.


Epoch 14/20 | Train Loss: 0.0389 | Valid Loss: 0.1276 | Accuracy: 98.25%
Epoch 15/20 | Train Loss: 0.0262 | Valid Loss: 0.1351 | Accuracy: 96.49%
Epoch 16/20 | Train Loss: 0.0309 | Valid Loss: 0.1339 | Accuracy: 96.49%
Epoch 17/20 | Train Loss: 0.0259 | Valid Loss: 0.1506 | Accuracy: 96.49%
Epoch 18/20 | Train Loss: 0.0188 | Valid Loss: 0.1622 | Accuracy: 96.49%
Epoch 19/20 | Train Loss: 0.0254 | Valid Loss: 0.1578 | Accuracy: 96.49%
Epoch 20/20 | Train Loss: 0.0142 | Valid Loss: 0.1639 | Accuracy: 97.37%

---WINNING SETUP---
Best Accuracy: 99.12%
  l1_neurons: 60
  l2_neurons: 13
  dropout_rate: 0.26009588933203365
  lr: 0.003792274993506615


In [10]:
best_params = study.best_params

print("Building final model with these parameters")
print(best_params)

final_model = CancerClassifier(
    n_features=30,
    n_units_l1=best_params["l1_neurons"],
    n_units_l2=best_params["l2_neurons"],
    dropout_rate=best_params["dropout_rate"]
).to(device)

final_optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params["lr"])
criterion = nn.BCEWithLogitsLoss()

print("\n---TRAINING FINAL MODEL---")
final_accuracy = train(
    model=final_model,
    optimizer=final_optimizer,
    criterion=criterion,
    train_loader=train_loader,
    valid_loader=valid_loader,
    n_epochs=50
)

torch.save(final_model.state_dict(), "cancer_predictor.pth")
print("\nModel weigths saved to 'best_cancer_model.pth'")

Building final model with these parameters
{'l1_neurons': 60, 'l2_neurons': 13, 'dropout_rate': 0.26009588933203365, 'lr': 0.003792274993506615}

---TRAINING FINAL MODEL---
Epoch 01/50 | Train Loss: 0.5788 | Valid Loss: 0.3426 | Accuracy: 96.49%
Epoch 02/50 | Train Loss: 0.2415 | Valid Loss: 0.0809 | Accuracy: 98.25%
Epoch 03/50 | Train Loss: 0.0953 | Valid Loss: 0.0520 | Accuracy: 98.25%
Epoch 04/50 | Train Loss: 0.0826 | Valid Loss: 0.0546 | Accuracy: 98.25%
Epoch 05/50 | Train Loss: 0.0777 | Valid Loss: 0.0517 | Accuracy: 98.25%
Epoch 06/50 | Train Loss: 0.0519 | Valid Loss: 0.0539 | Accuracy: 98.25%
Epoch 07/50 | Train Loss: 0.0569 | Valid Loss: 0.0524 | Accuracy: 98.25%
Epoch 08/50 | Train Loss: 0.0510 | Valid Loss: 0.0618 | Accuracy: 98.25%
Epoch 09/50 | Train Loss: 0.0500 | Valid Loss: 0.0569 | Accuracy: 98.25%
Epoch 10/50 | Train Loss: 0.0469 | Valid Loss: 0.0521 | Accuracy: 98.25%
Epoch 11/50 | Train Loss: 0.0426 | Valid Loss: 0.0635 | Accuracy: 97.37%
Epoch 12/50 | Train Loss

In [11]:
from torchsummary import summary

In [12]:
summary(final_model, (X.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 60]           1,860
              ReLU-2                   [-1, 60]               0
           Dropout-3                   [-1, 60]               0
            Linear-4                   [-1, 13]             793
              ReLU-5                   [-1, 13]               0
           Dropout-6                   [-1, 13]               0
            Linear-7                    [-1, 1]              14
Total params: 2,667
Trainable params: 2,667
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.01
Estimated Total Size (MB): 0.01
----------------------------------------------------------------
